In [1]:
!pip install sentence-transformers torch shap joblib pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
import joblib
import json

from sentence_transformers import SentenceTransformer

print("Loading dataset...")
df = pd.read_csv("fairlens_dataset_unstructured.csv")

print("Loading baseline model...")
baseline_model = joblib.load("biased_baseline_ats.pkl")

print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Rows:", len(df))
print("System Ready ✅")

Loading dataset...
Loading baseline model...


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Rows: 9544
System Ready ✅


In [5]:
import re
import pandas as pd
import numpy as np

# ---------------------------------------
# LOAD REAL DATASET
# ---------------------------------------

df = pd.read_csv("fairlens_dataset_unstructured.csv")

# ---------------------------------------
# FINAL CURATED TERM BANK
# ---------------------------------------

CANDIDATE_TERMS = [

    # FAIRNESS / DEMOGRAPHIC PROXIES
    "Female", "Male",
    "Tier 1", "Tier 2", "Tier 3",
    "Metro", "Non-Metro",

    # CORE TECHNICAL SKILLS
    "Python", "SQL", "Excel", "Power BI", "Tableau",
    "Machine Learning", "Data Science", "Deep Learning",
    "Statistics", "R", "SAS",
    "Java", "C++", "C", "JavaScript",
    "AWS", "Azure", "Cloud", "Docker",

    # DATA / BUSINESS TOOLS
    "Analytics", "Data Analysis", "Business Analysis",
    "Visualization", "Dashboard", "Reporting",

    # JOB ROLES
    "Data Analyst",
    "Business Analyst",
    "Software Developer",
    "Machine Learning Engineer",
    "Engineer",
    "Developer",
    "Manager",
    "Consultant",
    "Accountant",
    "Auditor",

    # EDUCATION SIGNALS
    "B.Tech", "B.E", "B.Sc", "M.Sc",
    "MBA", "Bachelor", "Master",
    "Computer Science",
    "Mathematics",
    "Engineering",

    # CERTIFICATIONS
    "Google Data Analytics",
    "Certified",
    "Certification",

    # LANGUAGE / COMMUNICATION
    "English", "Hindi",

    # EXPERIENCE SIGNALS
    "Experienced",
    "Fresher",
    "Senior",
    "Lead",
    "Internship"
]

# ---------------------------------------
# BASELINE MODEL PREDICTION
# ---------------------------------------

def predict_baseline(resume_text):
    vec = embed_model.encode([resume_text])
    score = float(baseline_model.predict(vec)[0])
    return round(score, 4)

# ---------------------------------------
# STRICT TERM MATCHING
# ---------------------------------------

def term_present(text, term):
    pattern = r"\b" + re.escape(term) + r"\b"
    return re.search(pattern, text, flags=re.IGNORECASE) is not None

# ---------------------------------------
# SAFE TERM REMOVAL
# ---------------------------------------

def remove_term(text, term):
    pattern = r"\b" + re.escape(term) + r"\b"
    updated = re.sub(pattern, "", text, flags=re.IGNORECASE)
    updated = re.sub(r"\s+", " ", updated).strip()
    return updated

# ---------------------------------------
# BASELINE EXPLAINABILITY ENGINE
# ---------------------------------------

def explain_baseline(resume_text, top_k=8):

    base_score = predict_baseline(resume_text)
    impacts = []

    for term in CANDIDATE_TERMS:

        if term_present(resume_text, term):

            modified_text = remove_term(resume_text, term)
            new_score = predict_baseline(modified_text)

            impact = round(base_score - new_score, 4)

            impacts.append({
                "word": term,
                "impact": impact
            })

    # Sort strongest impacts first
    impacts = sorted(
        impacts,
        key=lambda x: abs(x["impact"]),
        reverse=True
    )[:top_k]

    return {
        "model_name": "Legacy ATS",
        "predicted_score": base_score,
        "top_impact_features": impacts
    }

print("Baseline Explainability Engine Ready ✅")

Baseline Explainability Engine Ready ✅


In [6]:
sample_resume = """
Alumni of Mumbai University (2021) with a B.Tech in Computer Science.
Expert in Python, SQL, Power BI, Excel, Machine Learning.
Certified in Google Data Analytics.
Professional background includes time at Infosys as a Data Analyst.
Experienced analytics professional seeking growth opportunities.
Languages spoken: English, Hindi.
"""

result = explain_baseline(sample_resume)

import json
print(json.dumps(result, indent=2))

{
  "model_name": "Legacy ATS",
  "predicted_score": 0.612,
  "top_impact_features": [
    {
      "word": "Python",
      "impact": -0.0146
    },
    {
      "word": "B.Tech",
      "impact": 0.0111
    },
    {
      "word": "Google Data Analytics",
      "impact": -0.0082
    },
    {
      "word": "Hindi",
      "impact": -0.0056
    },
    {
      "word": "Computer Science",
      "impact": -0.0049
    },
    {
      "word": "SQL",
      "impact": -0.0048
    },
    {
      "word": "English",
      "impact": -0.0042
    },
    {
      "word": "Machine Learning",
      "impact": -0.0039
    }
  ]
}


In [10]:
import json
import random
from collections import defaultdict
import numpy as np

# -----------------------------------
# FULL DATASET PREDICTIONS
# -----------------------------------

texts = df["Raw_Resume_Text"].astype(str).tolist()

print("Encoding all resumes...")
X = embed_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True
)

print("Predicting scores...")
scores = baseline_model.predict(X)

df["Baseline_Score"] = scores

threshold = 0.70

# -----------------------------------
# CREATE REPORT OBJECT
# -----------------------------------

report = {
    "model_name": "Legacy ATS",
    "rows_evaluated": int(len(df)),
    "mean_score": round(float(df["Baseline_Score"].mean()), 4),
    "median_score": round(float(df["Baseline_Score"].median()), 4),
    "min_score": round(float(df["Baseline_Score"].min()), 4),
    "max_score": round(float(df["Baseline_Score"].max()), 4),
    "selected_at_0_70_percent": round(
        float((df["Baseline_Score"] >= threshold).mean() * 100), 2
    )
}

# -----------------------------------
# 300 EXPLANATION SAMPLE
# -----------------------------------

random.seed(42)

sample_size = min(300, len(df))
sample_idxs = random.sample(range(len(df)), sample_size)

term_totals = defaultdict(float)
sample_outputs = []

print("Running explanations on 300 sampled resumes...")

for idx in sample_idxs:

    txt = df.iloc[idx]["Raw_Resume_Text"]
    exp = explain_baseline(txt)

    if len(sample_outputs) < 10:
        sample_outputs.append(exp)

    for item in exp["top_impact_features"]:
        term_totals[item["word"]] += item["impact"]

# -----------------------------------
# AGGREGATE TERMS
# -----------------------------------

top_terms = sorted(
    term_totals.items(),
    key=lambda x: abs(x[1]),
    reverse=True
)[:15]

report["rows_explained"] = sample_size
report["top_terms_overall"] = [
    {
        "word": k,
        "net_impact": round(v, 4)
    }
    for k, v in top_terms
]

report["sample_explanations"] = sample_outputs

# -----------------------------------
# SAVE
# -----------------------------------

with open("baseline_consolidated_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved baseline_consolidated_report.json")
print(json.dumps(report, indent=2)[:5000])

Encoding all resumes...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]

Predicting scores...
Running explanations on 300 sampled resumes...
Saved baseline_consolidated_report.json
{
  "model_name": "Legacy ATS",
  "rows_evaluated": 9544,
  "mean_score": 0.6609,
  "median_score": 0.6622,
  "min_score": 0.3172,
  "max_score": 0.8803,
  "selected_at_0_70_percent": 34.14,
  "rows_explained": 300,
  "top_terms_overall": [
    {
      "word": "Machine Learning",
      "net_impact": -3.1378
    },
    {
      "word": "Python",
      "net_impact": -0.894
    },
    {
      "word": "B.Tech",
      "net_impact": -0.8822
    },
    {
      "word": "Data Science",
      "net_impact": -0.749
    },
    {
      "word": "Machine Learning Engineer",
      "net_impact": -0.7358
    },
    {
      "word": "Bachelor",
      "net_impact": 0.708
    },
    {
      "word": "Engineering",
      "net_impact": 0.5862
    },
    {
      "word": "Manager",
      "net_impact": 0.5239
    },
    {
      "word": "Fresher",
      "net_impact": -0.5198
    },
    {
      "word": "Account

In [7]:
!unzip -o "fairlens_clean_model.pth.zip"

import os
print(os.listdir())

Archive:  fairlens_clean_model.pth.zip
 extracting: fairlens_clean_model/data.pkl  
 extracting: fairlens_clean_model/.format_version  
 extracting: fairlens_clean_model/.storage_alignment  
 extracting: fairlens_clean_model/byteorder  
 extracting: fairlens_clean_model/data/0  
 extracting: fairlens_clean_model/data/1  
 extracting: fairlens_clean_model/data/2  
 extracting: fairlens_clean_model/data/3  
 extracting: fairlens_clean_model/data/4  
 extracting: fairlens_clean_model/data/5  
 extracting: fairlens_clean_model/data/6  
 extracting: fairlens_clean_model/data/7  
 extracting: fairlens_clean_model/data/8  
 extracting: fairlens_clean_model/data/9  
 extracting: fairlens_clean_model/data/10  
 extracting: fairlens_clean_model/data/11  
 extracting: fairlens_clean_model/data/12  
 extracting: fairlens_clean_model/data/13  
 extracting: fairlens_clean_model/data/14  
 extracting: fairlens_clean_model/data/15  
 extracting: fairlens_clean_model/data/16  
 extracting: fairlens_cle

In [12]:

import os

print(os.listdir("fairlens_clean_model"))

['byteorder', '.format_version', '.data', 'data', 'data.pkl', '.storage_alignment', 'version']


In [15]:
!cp "fairlens_clean_model.pth.zip" fairlens_clean_model.pth


In [16]:
import torch
import pandas as pd

from sentence_transformers import SentenceTransformer
from dann_model import FairLensDANN

df = pd.read_csv("fairlens_dataset_unstructured.csv")

def discover_protected_columns(df, exclude=['Raw_Resume_Text', 'matched_score', 'protected_group']):
    cols = []
    for col in df.columns:
        if col not in exclude and df[col].nunique() < 10:
            if isinstance(df[col].dropna().iloc[0], str):
                cols.append(col)
    return cols

protected_columns = discover_protected_columns(df)

print("Protected columns:", protected_columns)

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

fair_model = FairLensDANN(
    protected_columns=protected_columns,
    input_dim=384,
    hidden_dim=512
)

state = torch.load(
    "fairlens_clean_model.pth",
    map_location="cpu",
    weights_only=False
)

fair_model.load_state_dict(state)
fair_model.eval()

print("FairLens model loaded successfully ✅")

Protected columns: ['gender', 'age_group', 'college_tier', 'region']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FairLens model loaded successfully ✅


In [17]:
def predict_fairlens(resume_text):
    vec = embed_model.encode([resume_text])
    x = torch.tensor(vec, dtype=torch.float32)

    with torch.no_grad():
        score, _ = fair_model(x, alpha=0.0)

    return round(float(score.numpy().flatten()[0]), 4)

sample_resume = """
Alumni of Mumbai University (2021) with a B.Tech in Computer Science.
Expert in Python, SQL, Power BI, Excel, Machine Learning.
Certified in Google Data Analytics.
Professional background includes time at Infosys as a Data Analyst.
Experienced analytics professional seeking growth opportunities.
Languages spoken: English, Hindi.
"""

print("FairLens Score:", predict_fairlens(sample_resume))

FairLens Score: 0.6359


In [18]:
import re

# -----------------------------------
# TERM MATCHING
# -----------------------------------

def term_present(text, term):
    pattern = r"\b" + re.escape(term) + r"\b"
    return re.search(pattern, text, flags=re.IGNORECASE) is not None

def remove_term(text, term):
    pattern = r"\b" + re.escape(term) + r"\b"
    updated = re.sub(pattern, "", text, flags=re.IGNORECASE)
    updated = re.sub(r"\s+", " ", updated).strip()
    return updated

# -----------------------------------
# FAIRLENS EXPLAINER
# -----------------------------------

def explain_fairlens(resume_text, top_k=8):

    base_score = predict_fairlens(resume_text)
    impacts = []

    for term in CANDIDATE_TERMS:

        if term_present(resume_text, term):

            modified = remove_term(resume_text, term)
            new_score = predict_fairlens(modified)

            impact = round(base_score - new_score, 4)

            impacts.append({
                "word": term,
                "impact": impact
            })

    impacts = sorted(
        impacts,
        key=lambda x: abs(x["impact"]),
        reverse=True
    )[:top_k]

    return {
        "model_name": "FairLens DANN",
        "predicted_score": base_score,
        "top_impact_features": impacts
    }

print("FairLens Explainability Ready ✅")

FairLens Explainability Ready ✅


In [19]:
result = explain_fairlens(sample_resume)

import json
print(json.dumps(result, indent=2))

{
  "model_name": "FairLens DANN",
  "predicted_score": 0.6359,
  "top_impact_features": [
    {
      "word": "Analytics",
      "impact": 0.0141
    },
    {
      "word": "Computer Science",
      "impact": 0.0091
    },
    {
      "word": "Data Analyst",
      "impact": 0.0087
    },
    {
      "word": "Python",
      "impact": -0.0077
    },
    {
      "word": "Machine Learning",
      "impact": 0.0052
    },
    {
      "word": "Experienced",
      "impact": 0.0039
    },
    {
      "word": "B.Tech",
      "impact": -0.0037
    },
    {
      "word": "Power BI",
      "impact": 0.0029
    }
  ]
}


In [20]:
import json
import random
import numpy as np
from collections import defaultdict

# -----------------------------------
# FULL DATASET FAIRLENS REPORT
# -----------------------------------

texts = df["Raw_Resume_Text"].astype(str).tolist()

print("Encoding all resumes...")
X = embed_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True
)

print("Predicting FairLens scores...")

scores = []

for i in range(0, len(X), 128):
    batch = torch.tensor(X[i:i+128], dtype=torch.float32)

    with torch.no_grad():
        preds, _ = fair_model(batch, alpha=0.0)

    scores.extend(preds.numpy().flatten().tolist())

df["FairLens_Score"] = scores

threshold = 0.70

# -----------------------------------
# REPORT HEADER
# -----------------------------------

report = {
    "model_name": "FairLens DANN",
    "rows_evaluated": int(len(df)),
    "mean_score": round(float(df["FairLens_Score"].mean()), 4),
    "median_score": round(float(df["FairLens_Score"].median()), 4),
    "min_score": round(float(df["FairLens_Score"].min()), 4),
    "max_score": round(float(df["FairLens_Score"].max()), 4),
    "selected_at_0_70_percent": round(
        float((df["FairLens_Score"] >= threshold).mean() * 100), 2
    )
}

# -----------------------------------
# 300 EXPLANATION SAMPLE
# -----------------------------------

random.seed(42)

sample_size = min(300, len(df))
sample_idxs = random.sample(range(len(df)), sample_size)

term_totals = defaultdict(float)
sample_outputs = []

print("Running explanations on 300 sampled resumes...")

for idx in sample_idxs:

    txt = df.iloc[idx]["Raw_Resume_Text"]
    exp = explain_fairlens(txt)

    if len(sample_outputs) < 10:
        sample_outputs.append(exp)

    for item in exp["top_impact_features"]:
        term_totals[item["word"]] += item["impact"]

# -----------------------------------
# TOP GLOBAL TERMS
# -----------------------------------

top_terms = sorted(
    term_totals.items(),
    key=lambda x: abs(x[1]),
    reverse=True
)[:15]

report["rows_explained"] = sample_size

report["top_terms_overall"] = [
    {
        "word": k,
        "net_impact": round(v, 4)
    }
    for k, v in top_terms
]

report["sample_explanations"] = sample_outputs

# -----------------------------------
# SAVE JSON
# -----------------------------------

with open("fairlens_consolidated_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved fairlens_consolidated_report.json")
print(json.dumps(report, indent=2)[:5000])

Encoding all resumes...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]

Predicting FairLens scores...
Running explanations on 300 sampled resumes...
Saved fairlens_consolidated_report.json
{
  "model_name": "FairLens DANN",
  "rows_evaluated": 9544,
  "mean_score": 0.6559,
  "median_score": 0.6496,
  "min_score": 0.3699,
  "max_score": 0.8871,
  "selected_at_0_70_percent": 26.81,
  "rows_explained": 300,
  "top_terms_overall": [
    {
      "word": "Machine Learning",
      "net_impact": -1.2149
    },
    {
      "word": "Certified",
      "net_impact": 0.7096
    },
    {
      "word": "Python",
      "net_impact": -0.3612
    },
    {
      "word": "Developer",
      "net_impact": -0.3368
    },
    {
      "word": "Bachelor",
      "net_impact": 0.3076
    },
    {
      "word": "Accountant",
      "net_impact": -0.2946
    },
    {
      "word": "Data Science",
      "net_impact": -0.2898
    },
    {
      "word": "Fresher",
      "net_impact": -0.2681
    },
    {
      "word": "Engineering",
      "net_impact": 0.2658
    },
    {
      "word": "Ma

In [21]:
import json

# -----------------------------------
# LOAD BOTH REPORTS
# -----------------------------------

with open("baseline_consolidated_report.json", "r") as f:
    baseline = json.load(f)

with open("fairlens_consolidated_report.json", "r") as f:
    fairlens = json.load(f)

# -----------------------------------
# EXTRACT TOP TERMS
# -----------------------------------

baseline_terms = [x["word"] for x in baseline["top_terms_overall"][:10]]
fairlens_terms = [x["word"] for x in fairlens["top_terms_overall"][:10]]

# -----------------------------------
# BUILD COMPARISON REPORT
# -----------------------------------

comparison = {
    "models_compared": [
        baseline["model_name"],
        fairlens["model_name"]
    ],

    "dataset_rows": baseline["rows_evaluated"],

    "threshold_used": 0.70,

    "metrics": {
        "baseline_mean_score": baseline["mean_score"],
        "fairlens_mean_score": fairlens["mean_score"],

        "baseline_selection_rate_percent":
            baseline["selected_at_0_70_percent"],

        "fairlens_selection_rate_percent":
            fairlens["selected_at_0_70_percent"],

        "selection_rate_difference_percent":
            round(
                fairlens["selected_at_0_70_percent"]
                - baseline["selected_at_0_70_percent"],
                2
            )
    },

    "explainability_comparison": {
        "baseline_top_terms": baseline_terms,
        "fairlens_top_terms": fairlens_terms
    },

    "summary_findings": [
        "Both models were evaluated on the full 9544-row dataset.",
        "Both models used 300 sampled resumes for explanation audits.",
        "Baseline model showed stronger irregular penalties on certain technical skills.",
        "FairLens model produced a different score distribution and more selective threshold behavior.",
        "FairLens may require threshold recalibration for deployment fairness targets."
    ]
}

# -----------------------------------
# SAVE
# -----------------------------------

with open("comparison_report.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("Saved comparison_report.json")
print(json.dumps(comparison, indent=2))

Saved comparison_report.json
{
  "models_compared": [
    "Legacy ATS",
    "FairLens DANN"
  ],
  "dataset_rows": 9544,
  "threshold_used": 0.7,
  "metrics": {
    "baseline_mean_score": 0.6609,
    "fairlens_mean_score": 0.6559,
    "baseline_selection_rate_percent": 34.14,
    "fairlens_selection_rate_percent": 26.81,
    "selection_rate_difference_percent": -7.33
  },
  "explainability_comparison": {
    "baseline_top_terms": [
      "Machine Learning",
      "Python",
      "B.Tech",
      "Data Science",
      "Machine Learning Engineer",
      "Bachelor",
      "Engineering",
      "Manager",
      "Fresher",
      "Accountant"
    ],
    "fairlens_top_terms": [
      "Machine Learning",
      "Certified",
      "Python",
      "Developer",
      "Bachelor",
      "Accountant",
      "Data Science",
      "Fresher",
      "Engineering",
      "Machine Learning Engineer"
    ]
  },
  "summary_findings": [
    "Both models were evaluated on the full 9544-row dataset.",
    "Both m